# 📱 Telecom Customer Churn & Revenue Intelligence
**End-to-end analytics notebook** – data quality → EDA → churn model → revenue intelligence.

Run top-to-bottom. Each section calls the production script from `../python/` so the notebook and the pipeline never drift apart, then displays the key outputs inline.

In [ ]:
import sys, pandas as pd
from pathlib import Path
from IPython.display import Image, display
ROOT = Path('..').resolve(); sys.path.append(str(ROOT / 'python'))
CH, REP = ROOT / 'outputs/charts', ROOT / 'outputs/reports'
pd.set_option('display.max_columns', 40)
def show(*names):
    for n in names: display(Image(filename=str(CH / n)))

## 1. Generate data (Jan-2024 → Dec-2025, 8,000 subscribers)

In [ ]:
%run ../python/01_generate_data.py

In [ ]:
raw = pd.read_csv(ROOT / 'data/raw/customers.csv')
raw.head()

## 2. Data cleaning & quality audit

In [ ]:
%run ../python/02_data_cleaning.py

In [ ]:
pd.read_csv(REP / 'data_quality_report.csv')

## 3. Exploratory data analysis – what drives churn?

In [ ]:
%run ../python/03_eda.py

In [ ]:
show('09_monthly_churn_trend.png', '01_churn_by_contract.png', '02_churn_by_tenure.png', '03_churn_by_circle.png')

In [ ]:
show('06_churn_by_tickets.png', '10_churn_reasons.png', '11_drop_rate_distribution.png', '12_feature_correlation.png')

In [ ]:
pd.read_csv(REP / 'segment_churn_analysis.csv').sort_values('lift_vs_avg', ascending=False).head(10)

## 4. Churn prediction model
* 3 models compared with 5-fold stratified CV (LogReg, Random Forest, Gradient Boosting)
* Leakage controls: 'Unpaid' final invoice & bill volatility excluded; active customers scored **out-of-fold**
* Threshold chosen on a **profit curve** (offer cost ₹300, save rate 30%, 40% margin, 12 months saved)

In [ ]:
%run ../python/04_churn_model.py

In [ ]:
display(pd.read_csv(REP / 'model_comparison.csv')); display(pd.read_csv(REP / 'final_model_metrics.csv'))

In [ ]:
show('13_roc_curves.png', '14_profit_curve.png', '15_confusion_matrix.png', '16_feature_importance.png')

In [ ]:
pd.read_csv(REP / 'churn_predictions.csv').head(15)

## 5. Revenue intelligence – MRR, ARPU, cohorts, CLV, revenue at risk

In [ ]:
%run ../python/05_revenue_intelligence.py

In [ ]:
show('17_mrr_trend.png', '18_arpu_trend.png', '19_subscriber_movement.png', '20_cohort_retention.png')

In [ ]:
show('21_clv_by_segment.png', '22_revenue_pareto.png', '23_revenue_at_risk_by_circle.png')

## 6. Build Power BI star schema & load SQL database (runs all 30 SQL queries)

In [ ]:
%run ../python/06_build_powerbi_model.py
%run ../python/07_load_sql.py

In [ ]:
pd.read_csv(ROOT / 'outputs/sql_results/Q12.csv')   # tariff-hike impact

## 7. Recommendations
1. **Contract migration** – move month-to-month users to 12-month plans (churn ~44% → ~15%).
2. **First-90-days onboarding** programme – 0-6 month tenure churns at ~48%.
3. **Network capex priority**: Uttar Pradesh & West Bengal (highest drop rates & churn).
4. **Ticket-triggered save desk** – customers with >0.5 tickets/month churn at ~1.8× average.
5. **Price-hike playbook** – targeted grandfathering for high-CLV users before tariff changes.